In [0]:
# Tabla 1: Distribución de calidad
from pyspark.sql.functions import count, avg, round as spark_round

df = spark.table("wine_quality_clean")

quality_dist = df.groupBy("wine_type", "quality_category", "quality") \
    .agg(count("*").alias("cantidad")) \
    .orderBy("wine_type", "quality")

quality_dist.write.format("delta").mode("overwrite").saveAsTable("dash_quality_dist")

In [0]:
# Tabla 2: Indicadores KPI globales
from pyspark.sql.functions import avg, count, round as spark_round

kpi = df.agg(
    count("*").alias("total_vinos"),
    spark_round(avg("quality"), 2).alias("calidad_promedio"),
    spark_round(avg("alcohol"), 2).alias("alcohol_promedio"),
    spark_round(avg("volatile_acidity"), 3).alias("acidez_volatil_promedio")
)

kpi.write.format("delta").mode("overwrite").saveAsTable("dash_kpi")

In [0]:
# Tabla 3: Químicos promedio por categoría y tipo
chemical_avg = df.groupBy("wine_type", "quality_category").agg(
    spark_round(avg("alcohol"), 2).alias("avg_alcohol"),
    spark_round(avg("volatile_acidity"), 3).alias("avg_volatile_acidity"),
    spark_round(avg("sulphates"), 3).alias("avg_sulphates"),
    spark_round(avg("citric_acid"), 3).alias("avg_citric_acid"),
    spark_round(avg("pH"), 2).alias("avg_pH"),
    count("*").alias("n")
)

chemical_avg.write.format("delta").mode("overwrite").saveAsTable("dash_chemical_avg")

In [0]:
# Tabla 4: Importancia de variables del modelo ML
import pandas as pd

# Recuperar importancias
predictions_df = spark.table("wine_ml_predictions")
print(f"Predicciones disponibles: {predictions_df.count()}")
display(predictions_df.groupBy("quality", "prediction").count())

print("✅ Tablas de dashboard preparadas.")